# Day 13: SystemVerilog for Design

## Pre-Class Videos (~45 minutes total)

| # | Segment | Duration | File |
|---|---------|----------|------|
| 1 | Why SystemVerilog? | ~8 min | `d13_s1_why_systemverilog.html` |
| 2 | `logic` — One Type to Rule Them All | ~10 min | `d13_s2_logic_type.html` |
| 3 | Intent-Based Always Blocks | ~12 min | `d13_s3_intent_based_always.html` |
| 4 | `enum`, `struct`, and `package` | ~15 min | `d13_s4_enum_struct_package.html` |

## Code Examples

| File | Description |
|------|-------------|
| `code/day13_ex01_uart_tx_sv.sv` | UART TX refactored: logic, always_ff, always_comb, enum states |
| `code/day13_ex02_alu_sv.sv` | ALU refactored: always_comb, enum opcodes, self-checking TB |

## Diagrams

| File | Description |
|------|-------------|
| `diagrams/d13_safety_net.svg` | Safety comparison: Verilog risks vs. SV protections |

## Key Concepts
- `logic` replaces `wire` + `reg` (single-driver restriction)
- `always_ff`: enforces clock edge, warns on blocking assignments
- `always_comb`: auto-sensitivity, **errors on latch inference**
- `enum`: type-safe FSM states, `.name()` for debug, zero cost
- `struct packed`: grouped signals with dot notation, synthesizable
- `package`: shared types/constants across modules
- Toolchain: `iverilog -g2012`, `yosys read_verilog -sv`

## Directory Structure

```
day13_systemverilog_for_design/
├── d13_s1_why_systemverilog.html
├── d13_s2_logic_type.html
├── d13_s3_intent_based_always.html
├── d13_s4_enum_struct_package.html
├── code/
│   ├── day13_ex01_uart_tx_sv.sv
│   └── day13_ex02_alu_sv.sv
├── diagrams/
│   └── d13_safety_net.svg
├── day13_quiz.md
└── day13_readme.md
```

---
## Code Examples

### `day13_ex01_uart_tx_sv.sv`

```verilog
// =============================================================================
// day13_ex01_uart_tx_sv.sv — UART TX Refactored in SystemVerilog
// Day 13: SystemVerilog for Design
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Same functionality as day11_ex01_uart_tx.v, refactored with:
//   - logic (replaces wire/reg)
//   - always_ff / always_comb (intent-based)
//   - typedef enum (type-safe FSM states)
//   - Dropped r_ prefix (always_ff declares register intent)
// =============================================================================
// Build:  iverilog -g2012 -DSIMULATION -o sim day13_ex01_uart_tx_sv.sv && vvp sim
// Synth:  yosys -p "read_verilog -sv day13_ex01_uart_tx_sv.sv; synth_ice40 -top uart_tx_sv"
// =============================================================================

module uart_tx_sv #(
    `ifdef SIMULATION
    parameter int CLK_FREQ  = 1_000,
    parameter int BAUD_RATE = 100
    `else
    parameter int CLK_FREQ  = 25_000_000,
    parameter int BAUD_RATE = 115_200
    `endif
)(
    input  logic       i_clk,
    input  logic       i_reset,
    input  logic       i_valid,
    input  logic [7:0] i_data,
    output logic       o_busy,
    output logic       o_tx
);

    // ---- Parameters ----
    localparam int CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;
    localparam int CNT_W        = $clog2(CLKS_PER_BIT);

    // ---- State Type (enum replaces localparam) ----
    typedef enum logic [1:0] {
        S_IDLE  = 2'b00,
        S_START = 2'b01,
        S_DATA  = 2'b10,
        S_STOP  = 2'b11
    } uart_state_t;

    uart_state_t state, next_state;

    // ---- Datapath Registers ----
    logic [CNT_W-1:0]  baud_cnt;
    logic [2:0]         bit_idx;
    logic [9:0]         shift;

    logic baud_tick;
    assign baud_tick = (baud_cnt == CNT_W'(CLKS_PER_BIT - 1));

    // ============================================================
    // Baud Counter — always_ff enforces sequential semantics
    // ============================================================
    always_ff @(posedge i_clk) begin
        if (i_reset || state == S_IDLE)
            baud_cnt <= '0;
        else if (baud_tick)
            baud_cnt <= '0;
        else
            baud_cnt <= baud_cnt + 1;
    end

    // ============================================================
    // Block 1 — State Register (always_ff)
    // ============================================================
    always_ff @(posedge i_clk) begin
        if (i_reset)
            state <= S_IDLE;
        else
            state <= next_state;
    end

    // ============================================================
    // Block 2 — Next-State Logic (always_comb catches latches!)
    // ============================================================
    always_comb begin
        next_state = state;     // default: hold (prevents latch)

        case (state)
            S_IDLE:
                if (i_valid)
                    next_state = S_START;

            S_START:
                if (baud_tick)
                    next_state = S_DATA;

            S_DATA:
                if (baud_tick && bit_idx == 3'd7)
                    next_state = S_STOP;

            S_STOP:
                if (baud_tick)
                    next_state = S_IDLE;

            default: next_state = S_IDLE;
        endcase
    end

    // ---- Shift Register & Bit Counter ----
    always_ff @(posedge i_clk) begin
        if (i_reset) begin
            shift   <= 10'h3FF;
            bit_idx <= '0;
        end else if (state == S_IDLE && i_valid) begin
            shift   <= {1'b1, i_data, 1'b0};  // {stop, data, start}
            bit_idx <= '0;
        end else if (baud_tick) begin
            shift <= {1'b1, shift[9:1]};
            if (state == S_DATA)
                bit_idx <= bit_idx + 1;
        end
    end

    // ---- Output Logic (always_comb) ----
    always_comb begin
        o_tx   = (state == S_IDLE) ? 1'b1 : shift[0];
        o_busy = (state != S_IDLE);
    end

endmodule

// =============================================================================
// Self-Checking Testbench — Same protocol-aware capture as Day 11
// =============================================================================
`ifdef SIMULATION
module tb_uart_tx_sv;

    localparam int CLK_FREQ     = 1_000;
    localparam int BAUD_RATE    = 100;
    localparam int CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;

    logic       clk = 0, reset = 1, valid = 0;
    logic [7:0] data;
    logic       busy, tx;

    uart_tx_sv #(
        .CLK_FREQ(CLK_FREQ),
        .BAUD_RATE(BAUD_RATE)
    ) uut (
        .i_clk(clk), .i_reset(reset),
        .i_valid(valid), .i_data(data),
        .o_busy(busy), .o_tx(tx)
    );

    always #5 clk = ~clk;

    int test_count = 0, fail_count = 0;

    task automatic capture_and_check(input logic [7:0] expected, input string name);
        logic [7:0] captured;
    begin
        test_count++;

        @(negedge tx);  // start bit falling edge
        repeat (CLKS_PER_BIT / 2) @(posedge clk);  // center of start

        if (tx !== 1'b0) begin
            $error("FAIL: %s — start bit not 0", name);
            fail_count++;
        end

        for (int i = 0; i < 8; i++) begin
            repeat (CLKS_PER_BIT) @(posedge clk);
            captured[i] = tx;
        end

        repeat (CLKS_PER_BIT) @(posedge clk);
        if (tx !== 1'b1) begin
            $error("FAIL: %s — stop bit not 1", name);
            fail_count++;
        end

        if (captured !== expected) begin
            $error("FAIL: %s — expected %h, captured %h", name, expected, captured);
            fail_count++;
        end else
            $display("PASS: %s — TX byte %h ('%c') correct", name, captured, captured);
    end
    endtask

    initial begin
        $dumpfile("tb_uart_tx_sv.vcd");
        $dumpvars(0, tb_uart_tx_sv);

        $display("\n=== UART TX (SystemVerilog) Testbench ===\n");

        repeat(5) @(posedge clk); reset = 0;
        repeat(5) @(posedge clk);

        // Test 1: 'H'
        fork
            begin data = 8'h48; valid = 1; @(posedge clk); valid = 0; end
            begin capture_and_check(8'h48, "SV Byte 'H'"); end
        join
        repeat(CLKS_PER_BIT * 2) @(posedge clk);

        // Test 2: 0x00
        fork
            begin data = 8'h00; valid = 1; @(posedge clk); valid = 0; end
            begin capture_and_check(8'h00, "SV Byte 0x00"); end
        join
        repeat(CLKS_PER_BIT * 2) @(posedge clk);

        // Test 3: 0xFF
        fork
            begin data = 8'hFF; valid = 1; @(posedge clk); valid = 0; end
            begin capture_and_check(8'hFF, "SV Byte 0xFF"); end
        join

        $display("\n=== SUMMARY ===");
        $display("Tests: %0d  Passed: %0d  Failed: %0d",
                 test_count, test_count - fail_count, fail_count);
        if (fail_count == 0) $display("\n*** ALL TESTS PASSED ***\n");
        else                 $display("\n*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

### `day13_ex02_alu_sv.sv`

```verilog
// =============================================================================
// day13_ex02_alu_sv.sv — ALU Refactored in SystemVerilog
// Day 13: SystemVerilog for Design
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Demonstrates: logic, always_comb, enum for opcodes.
// Compared to Verilog version: stronger latch checking, typed opcodes.
// =============================================================================
// Build:  iverilog -g2012 -DSIMULATION -o sim day13_ex02_alu_sv.sv && vvp sim
// =============================================================================

module alu_sv #(
    parameter int WIDTH = 4
)(
    input  logic [WIDTH-1:0]  i_a,
    input  logic [WIDTH-1:0]  i_b,
    input  logic [1:0]        i_op,
    output logic [WIDTH-1:0]  o_result,
    output logic              o_carry,
    output logic              o_zero
);

    // ---- Opcode Enum ----
    typedef enum logic [1:0] {
        OP_ADD = 2'b00,
        OP_SUB = 2'b01,
        OP_AND = 2'b10,
        OP_OR  = 2'b11
    } alu_op_t;

    logic [WIDTH:0] full_result;   // extra bit for carry

    // always_comb: auto-sensitivity, catches latches
    always_comb begin
        full_result = '0;   // default prevents latch

        case (i_op)
            OP_ADD: full_result = {1'b0, i_a} + {1'b0, i_b};
            OP_SUB: full_result = {1'b0, i_a} - {1'b0, i_b};
            OP_AND: full_result = {1'b0, i_a & i_b};
            OP_OR:  full_result = {1'b0, i_a | i_b};
            default: full_result = '0;
        endcase
    end

    assign o_result = full_result[WIDTH-1:0];
    assign o_carry  = full_result[WIDTH];
    assign o_zero   = (o_result == '0);

endmodule

// =============================================================================
// Self-Checking Testbench
// =============================================================================
`ifdef SIMULATION
module tb_alu_sv;

    logic [3:0] a, b;
    logic [1:0] op;
    logic [3:0] result;
    logic       carry, zero;

    alu_sv #(.WIDTH(4)) uut (
        .i_a(a), .i_b(b), .i_op(op),
        .o_result(result), .o_carry(carry), .o_zero(zero)
    );

    int test_count = 0, fail_count = 0;

    task automatic check(
        input logic [3:0] ea, eb,
        input logic [1:0] eop,
        input logic [3:0] exp_res,
        input logic       exp_carry,
        input string      name
    );
    begin
        test_count++;
        a = ea; b = eb; op = eop;
        #1;
        if (result !== exp_res || carry !== exp_carry) begin
            $error("FAIL: %s — got %h c=%b, expected %h c=%b",
                   name, result, carry, exp_res, exp_carry);
            fail_count++;
        end else
            $display("PASS: %s — %h (carry=%b, zero=%b)", name, result, carry, zero);
    end
    endtask

    initial begin
        $display("\n=== ALU (SystemVerilog) Testbench ===\n");

        // ADD tests
        check(4'h3, 4'h5, 2'b00, 4'h8, 1'b0, "ADD 3+5=8");
        check(4'hF, 4'h1, 2'b00, 4'h0, 1'b1, "ADD F+1=0+carry");
        check(4'h0, 4'h0, 2'b00, 4'h0, 1'b0, "ADD 0+0=0 zero");

        // SUB tests
        check(4'h8, 4'h3, 2'b01, 4'h5, 1'b0, "SUB 8-3=5");
        check(4'h0, 4'h1, 2'b01, 4'hF, 1'b1, "SUB 0-1=F+borrow");

        // AND tests
        check(4'hF, 4'hA, 2'b10, 4'hA, 1'b0, "AND F&A=A");
        check(4'h5, 4'hA, 2'b10, 4'h0, 1'b0, "AND 5&A=0 zero");

        // OR tests
        check(4'h5, 4'hA, 2'b11, 4'hF, 1'b0, "OR 5|A=F");
        check(4'h0, 4'h0, 2'b11, 4'h0, 1'b0, "OR 0|0=0 zero");

        $display("\n=== SUMMARY ===");
        $display("Tests: %0d  Passed: %0d  Failed: %0d",
                 test_count, test_count - fail_count, fail_count);
        if (fail_count == 0) $display("\n*** ALL TESTS PASSED ***\n");
        else                 $display("\n*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

---
## Pre-Class Self-Check Quiz

**Q1:** What does the `logic` type replace? What is its one restriction?

<details><summary>Answer</summary>
`logic` replaces both `wire` and `reg`. It can be driven by `assign`, `always_ff`, or `always_comb`. Its one restriction: it can only have **one driver**. For multi-driver buses (rare in FPGA), use `wire`.
</details>

**Q2:** What happens if you accidentally create an incomplete `case` statement (missing a default) inside `always_comb`? How is this different from Verilog's `always @(*)`?

<details><summary>Answer</summary>
In `always_comb`, the compiler **errors** because a latch would be inferred — `always_comb` requires purely combinational logic. In Verilog's `always @(*)`, a latch is silently inferred with at most a synthesis warning that you might miss. This is the single biggest safety win of SystemVerilog.
</details>

**Q3:** Rewrite this Verilog FSM state declaration using SystemVerilog `enum`:
```verilog
localparam S_IDLE = 2'b00, S_RUN = 2'b01, S_DONE = 2'b10;
reg [1:0] state;
```

<details><summary>Answer</summary>

```systemverilog
typedef enum logic [1:0] {
    S_IDLE = 2'b00,
    S_RUN  = 2'b01,
    S_DONE = 2'b10
} my_state_t;

my_state_t state;
```
Bonus: `$display("State: %s", state.name());` prints the state name.
</details>

**Q4:** What is a `packed struct` and why is it synthesizable?

<details><summary>Answer</summary>
A `packed struct` stores its fields as a contiguous bit vector. For example, a struct with an 8-bit data field, a valid bit, and a busy bit is 10 bits wide. Because it's a flat bit vector, synthesis tools can map it directly to wires and registers — no different from a `logic [9:0]` bus, but with named fields for readability.
</details>